# CivilVerify · Defect finder (crack · cold joint · honeycomb)

Trains a small image model (MobileNetV3-Small, transfer learning) on **your** photos and exports `defects.tflite`
(LiteRT format, runs offline in the CivilVerify app).

**What the model gives the app for every photo**
1. a probability for each class: `crack`, `cold_joint`, `honeycomb`, `good`
2. a coarse **heat map** (8 x 8 cells) of where each class is strongest, which the app turns into a circle around the suspect area

**Important to understand**
Your photos are labelled for the *whole picture* ("this photo has a crack"), not with a box around the defect.
The model therefore learns *where* from the whole-picture label only (class activation map). The circle is a **good guide, not a surveyed outline**,
and it is coarser than a hand-drawn box. This notebook shows you held-out examples so you can judge it yourself.

**Before you start**
1. Runtime > Change runtime type > **GPU** (T4).
2. Upload `civilverify_dataset.zip` (folders `crack`, `cold_joint`, `honeycomb`, `good`). To add photos later, add them to those folders and rezip.
3. Run the cells top to bottom (about 20 to 30 minutes). The honest cross-validated results are printed *before* the final model is trained.

**Keeping results honest**
- Near-duplicate photos are grouped, so a copy of a training photo never lands in the test set.
- Fake watermarks, arrows, boxes and words such as "Cold joint" are drawn on random photos of **all** classes, so the model cannot learn "arrow or caption = this defect".
- All reported numbers come from grouped cross-validation on whole photos, not from the training data.

In [ ]:
import os, io, json, glob, random, shutil, zipfile, datetime
import numpy as np
from PIL import Image, ImageOps, ImageDraw, ImageFont, ImageFilter
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

SEED = 7
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG = 256            # model input size -> 8 x 8 heat map
N_AUG_TRAIN = 20     # augmented variants per training photo
N_AUG_EVAL  = 6      # variants per photo when testing (their average is the photo's score)
FOLDS = 5
RECALL_TARGET = 0.85 # per-defect share of photos we want to catch when choosing thresholds

## 1. Upload your photos

In [ ]:
from google.colab import files
up = files.upload()                       # choose civilverify_dataset.zip
zname = list(up.keys())[0]
shutil.rmtree("data", ignore_errors=True)
with zipfile.ZipFile(zname) as z: z.extractall("data")
print("extracted", zname)

## 2. Load photos and list the classes found

In [ ]:
EXT = (".jpg", ".jpeg", ".jfif", ".png", ".webp", ".bmp")
items = []   # (PIL image, class name, file name)
for root, _, fs in os.walk("data"):
    if "__MACOSX" in root: continue
    for f in sorted(fs):
        if f.lower().endswith(EXT) and not f.startswith("._"):
            cls = os.path.basename(root).strip().lower().replace(" ", "_")
            try:
                im = ImageOps.exif_transpose(Image.open(os.path.join(root, f))).convert("RGB")
            except Exception as e:
                print("skip", f, e); continue
            im.thumbnail((512, 512), Image.LANCZOS)
            items.append((im, cls, f))

classes = sorted({c for _, c, _ in items})
assert "good" in classes, "Need a folder named 'good' (photos with NO defect). Found: %s" % classes
DEFECTS = [c for c in classes if c != "good"]
cid = {c: i for i, c in enumerate(classes)}
y = np.array([cid[c] for _, c, _ in items])
print("classes:", classes)
for c in classes: print(f"  {c:12s} {int((y == cid[c]).sum())} photos")
if int((y == cid["good"]).sum()) < 0.6 * min(int((y == cid[c]).sum()) for c in DEFECTS):
    print("\nWARNING: you have far fewer 'good' photos than defect photos. The model will be biased towards flagging.")
    print("Add more good photos: plain concrete, plastered walls, painted walls, smooth formwork finish, floors, columns.")

## 3. Group near-duplicate photos
Duplicates (same photo saved twice, crops of one wall) are forced into the same fold.

In [ ]:
def dhash(im, s=8):
    g = np.asarray(im.convert("L").resize((s + 1, s), Image.LANCZOS), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).flatten()

H = [dhash(im) for im, _, _ in items]
par = list(range(len(items)))
def find(i):
    while par[i] != i:
        par[i] = par[par[i]]; i = par[i]
    return i
for i in range(len(items)):
    for j in range(i + 1, len(items)):
        if int((H[i] != H[j]).sum()) <= 8:
            if y[i] != y[j]: print("CHECK: near-identical photos in different classes:", items[i][2], items[j][2])
            par[find(i)] = find(j)
group = np.array([find(i) for i in range(len(items))])
print("photos:", len(items), "| independent groups:", len(set(group)))

## 4. Augmentation
Random crops, flips, brightness/contrast, blur, JPEG re-compression, and **fake watermarks, arrows, boxes and captions on all classes**.

In [ ]:
WM_TEXT = ["shutterstock", "iStock", "Credit: photographer", "pexels", "stock photo", "(c) 2025", "alamy"]
CAPTIONS = ["Cold Joint", "Crack", "Honeycomb", "Defect", "Note", "Column", "Slab", "Wall", "Visible line"]
def font(sz):
    for name in ["DejaVuSans.ttf", "DejaVuSans-Bold.ttf"]:
        try: return ImageFont.truetype(name, sz)
        except Exception: pass
    return ImageFont.load_default()

def add_watermark(im, rng):
    w, h = im.size
    layer = Image.new("RGBA", im.size, (0, 0, 0, 0)); d = ImageDraw.Draw(layer)
    f = font(int(rng.uniform(0.06, 0.16) * min(w, h)))
    txt = WM_TEXT[rng.integers(len(WM_TEXT))]
    col = (255, 255, 255, int(rng.uniform(40, 120))) if rng.random() < 0.6 else (60, 60, 60, int(rng.uniform(40, 120)))
    for _ in range(int(rng.integers(1, 4))):
        d.text((rng.uniform(0, 0.6) * w, rng.uniform(0, 0.8) * h), txt, font=f, fill=col)
    if rng.random() < 0.5: layer = layer.rotate(float(rng.uniform(-25, 25)), resample=Image.BICUBIC)
    return Image.alpha_composite(im.convert("RGBA"), layer).convert("RGB")

def add_markup(im, rng):
    # hand-made annotations seen on web photos: coloured captions, arrows, outline boxes. Added to ALL classes.
    w, h = im.size; d = ImageDraw.Draw(im)
    col = [(255, 220, 0), (255, 255, 255), (230, 30, 30), (20, 20, 20)][int(rng.integers(4))]
    if rng.random() < 0.6:
        d.text((rng.uniform(0.02, 0.6) * w, rng.uniform(0.02, 0.9) * h), CAPTIONS[int(rng.integers(len(CAPTIONS)))],
               font=font(int(rng.uniform(0.05, 0.11) * min(w, h))), fill=col, stroke_width=2, stroke_fill=(0, 0, 0))
    if rng.random() < 0.6:
        x0, y0 = rng.uniform(0.1, 0.9) * w, rng.uniform(0.1, 0.9) * h
        a = rng.uniform(0, 2 * np.pi); L = rng.uniform(0.1, 0.3) * min(w, h)
        x1, y1 = x0 + L * np.cos(a), y0 + L * np.sin(a)
        d.line([(x0, y0), (x1, y1)], fill=col, width=max(3, int(0.012 * min(w, h))))
        for da in (2.6, -2.6):
            d.line([(x1, y1), (x1 + 0.35 * L * np.cos(a + da), y1 + 0.35 * L * np.sin(a + da))], fill=col, width=max(3, int(0.012 * min(w, h))))
    if rng.random() < 0.3:
        bx0, by0 = rng.uniform(0.05, 0.5) * w, rng.uniform(0.05, 0.5) * h
        d.rectangle([bx0, by0, bx0 + rng.uniform(0.2, 0.5) * w, by0 + rng.uniform(0.2, 0.5) * h], outline=(230, 30, 30), width=max(3, int(0.01 * min(w, h))))
    return im

def augment(im, rng, train=True):
    w, h = im.size
    s = rng.uniform(0.5, 1.0) if train else rng.uniform(0.75, 1.0)
    ar = rng.uniform(0.8, 1.25)
    cw, ch = int(min(w, w * np.sqrt(s * ar))), int(min(h, h * np.sqrt(s / ar)))
    x0, y0 = int(rng.integers(0, w - cw + 1)), int(rng.integers(0, h - ch + 1))
    a = im.crop((x0, y0, x0 + cw, y0 + ch))
    if rng.random() < 0.5: a = ImageOps.mirror(a)
    if train and rng.random() < 0.4: a = a.rotate(int(rng.choice([90, 180, 270])), expand=True)
    if rng.random() < 0.4: a = add_watermark(a, rng)
    if rng.random() < 0.4: a = add_markup(a.copy(), rng)
    a = a.resize((IMG, IMG), Image.BICUBIC)
    arr = np.asarray(a, dtype=np.float32)
    arr = (arr - 128) * rng.uniform(0.75, 1.3) + 128 + rng.uniform(-30, 30)
    a = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    if rng.random() < 0.2: a = a.filter(ImageFilter.GaussianBlur(float(rng.uniform(0.3, 1.2))))
    if rng.random() < 0.3:
        b = io.BytesIO(); a.save(b, "JPEG", quality=int(rng.integers(40, 90))); a = Image.open(io.BytesIO(b.getvalue())).convert("RGB")
    return np.asarray(a, dtype=np.uint8)            # stored as uint8 to save memory; the model receives raw 0..255 floats

def make_set(idx, n, train, seed):
    rng = np.random.default_rng(seed)
    X, Y, src = [], [], []
    for i in idx:
        for _ in range(n):
            X.append(augment(items[i][0], rng, train)); Y.append(y[i]); src.append(i)
    return np.stack(X), np.array(Y), np.array(src)

def full_view(im):
    return np.asarray(im.resize((IMG, IMG), Image.BICUBIC), dtype=np.uint8)   # whole photo, no crop (same as the app)

import matplotlib.pyplot as plt
Xs, Ys, _ = make_set(list(range(0, len(items), max(1, len(items) // 8)))[:8], 1, True, 1)
fig, ax = plt.subplots(1, len(Xs), figsize=(2.2 * len(Xs), 2.4))
for k, a in enumerate(np.atleast_1d(ax)): a.imshow(Xs[k]); a.set_title(classes[Ys[k]], fontsize=8); a.axis("off")
plt.show()

## 5. Model
MobileNetV3-Small (ImageNet weights) with a **fully convolutional head**: a 1x1 convolution gives every one of the 8 x 8 cells a score for each class, and the photo's class probabilities are the average over cells.
That average is what the model is trained on; the per-cell scores are the heat map the app uses to draw the circle.

In [ ]:
def build(n_cls):
    base = tf.keras.applications.MobileNetV3Small(input_shape=(IMG, IMG, 3), include_top=False,
                                                  weights="imagenet", include_preprocessing=True)
    base.trainable = False
    inp = tf.keras.Input((IMG, IMG, 3), name="pixels")             # float32, raw 0..255
    x = base(inp, training=False)                                  # (8, 8, 576)
    x = tf.keras.layers.Dropout(0.3)(x)
    cam = tf.keras.layers.Conv2D(n_cls, 1, name="cam")(x)          # (8, 8, classes): per-cell class scores
    logits = tf.keras.layers.GlobalAveragePooling2D(name="pool")(cam)
    probs = tf.keras.layers.Softmax(name="probs")(logits)
    train_model = tf.keras.Model(inp, probs)
    export_model = tf.keras.Model(inp, [probs, cam])               # shares the same weights
    return train_model, export_model, base

class Seq(tf.keras.utils.Sequence):
    # feeds uint8 photos as float32 batches, with per-sample weights so that small classes count as much as large ones
    def __init__(self, X, Y, bs=32, shuffle=True):
        super().__init__()
        self.X, self.Y, self.bs, self.shuffle = X, Y, bs, shuffle
        cnt = np.bincount(Y, minlength=len(classes)).astype(np.float32)
        self.w = (len(Y) / (len(classes) * np.maximum(cnt, 1)))[Y].astype(np.float32)
        self.ix = np.arange(len(X)); self.on_epoch_end()
    def __len__(self): return int(np.ceil(len(self.X) / self.bs))
    def __getitem__(self, i):
        j = self.ix[i * self.bs:(i + 1) * self.bs]
        return self.X[j].astype(np.float32), self.Y[j], self.w[j]
    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.ix)

def fit(Xtr, Ytr, epochs_head=8, epochs_ft=8, verbose=0):
    tm, em, base = build(len(classes))
    tm.compile(tf.keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    tm.fit(Seq(Xtr, Ytr), epochs=epochs_head, verbose=verbose)
    base.trainable = True
    for l in base.layers[:-40]: l.trainable = False
    for l in base.layers:
        if isinstance(l, tf.keras.layers.BatchNormalization): l.trainable = False
    tm.compile(tf.keras.optimizers.Adam(2e-5), "sparse_categorical_crossentropy", metrics=["accuracy"])
    tm.fit(Seq(Xtr, Ytr), epochs=epochs_ft, verbose=verbose)
    return em

def predict(em, X):
    P, CAM = [], []
    for i in range(0, len(X), 64):
        p, c = em.predict(X[i:i + 64].astype(np.float32), verbose=0)
        P.append(p); CAM.append(c)
    return np.concatenate(P), np.concatenate(CAM)

## 6. Honest test: grouped cross-validation
Every photo is predicted by a model that **never saw it or its duplicates**. Read these numbers, not training accuracy.

In [ ]:
def grouped_folds(group, y, k, seed=0):
    rng = np.random.default_rng(seed)
    gs = list(set(group)); rng.shuffle(gs)
    gy = {g: int(np.bincount(y[group == g]).argmax()) for g in gs}
    folds = [[] for _ in range(k)]
    for c in sorted(set(gy.values())):
        for n, g in enumerate([g for g in gs if gy[g] == c]): folds[n % k].append(g)
    return [np.where(np.isin(group, f))[0] for f in folds]

def auc_rank(scores, labels):
    pos, neg = scores[labels == 1], scores[labels == 0]
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    return float(np.mean([(p > q) + 0.5 * (p == q) for p in pos for q in neg]))

oof = np.zeros((len(items), len(classes)))                         # out-of-fold probabilities per photo
oof_cam = np.zeros((len(items), IMG // 32, IMG // 32, len(classes)), dtype=np.float32)
for k, va in enumerate(grouped_folds(group, y, FOLDS)):
    tr = np.setdiff1d(np.arange(len(items)), va)
    if len(va) == 0 or len(set(y[tr])) < len(classes): continue
    Xtr, Ytr, _ = make_set(tr, N_AUG_TRAIN, True, 100 + k)
    Xva, _, sva = make_set(va, N_AUG_EVAL, False, 200 + k)
    em = fit(Xtr, Ytr)
    p, _ = predict(em, Xva)
    for i in va: oof[i] = p[sva == i].mean(axis=0)
    _, cc = predict(em, np.stack([full_view(items[i][0]) for i in va]))   # heat maps of held-out photos
    oof_cam[va] = cc
    print(f"fold {k + 1}/{FOLDS} done ({len(va)} photos tested)")
    del Xtr, Xva; tf.keras.backend.clear_session()

**Reading the results.** Rows are the true class, columns what the model said (highest probability).

In [ ]:
import pandas as pd
pred = oof.argmax(1)
cm = pd.DataFrame(np.zeros((len(classes), len(classes)), dtype=int), index=["true " + c for c in classes], columns=["said " + c for c in classes])
for t, p_ in zip(y, pred): cm.iloc[t, p_] += 1
print(cm.to_string()); print()
print("Share of photos where the top class was right: %.0f%%" % (100 * (pred == y).mean()))
aucs = {c: auc_rank(oof[:, cid[c]], (y == cid[c]).astype(int)) for c in classes}
print("AUC per class (1.0 perfect, 0.5 coin toss):", {c: round(a, 3) for c, a in aucs.items()})

**Choosing thresholds.** The app flags a defect when its probability is above that defect's threshold.
For each defect we take the highest threshold that still catches `RECALL_TARGET` of its photos (or the best balance if that is not reachable).

In [ ]:
def counts(th):
    # th: dict defect -> threshold. A photo is "flagged as c" when p_c >= th_c (a photo can carry several flags).
    fl = {c: oof[:, cid[c]] >= th[c] for c in DEFECTS}
    rows = {}
    for c in DEFECTS:
        isC = y == cid[c]
        rows[c] = dict(caught=int((fl[c] & isC).sum()), total=int(isC.sum()),
                       on_good=int((fl[c] & (y == cid["good"])).sum()),
                       on_other_defects=int((fl[c] & ~isC & (y != cid["good"])).sum()))
    anyflag = np.any([fl[c] for c in DEFECTS], axis=0)
    good_flagged = int((anyflag & (y == cid["good"])).sum())
    return rows, good_flagged

THR = {}
for c in DEFECTS:
    s, isC = oof[:, cid[c]], (y == cid[c])
    best, bj = None, -9
    for t in np.arange(0.2, 0.96, 0.01):
        tpr = (s[isC] >= t).mean(); fpr = (s[~isC] >= t).mean()
        if tpr >= RECALL_TARGET: best = float(t)                      # highest threshold that still meets the target
        if tpr - fpr > bj: bj, bt = tpr - fpr, float(t)
    THR[c] = round(best if best is not None else bt, 2)
rows, good_flagged = counts(THR)
print("thresholds:", THR)
for c, r in rows.items():
    print(f"{c:11s} caught {r['caught']}/{r['total']} | good photos flagged as {c}: {r['on_good']} | other-defect photos also flagged as {c}: {r['on_other_defects']}")
ngood = int((y == cid["good"]).sum())
print(f"\nGood photos flagged with ANY defect: {good_flagged}/{ngood}")

**Held-out heat-map examples.** These photos were *not* used to train the model that drew these circles. The circle logic is exactly the one used in the app:
cells where the defect's score is within 60 % of its peak, joined to the peak cell, then boxed and rounded.
Judge for yourself whether the circles sit on the defect.

In [ ]:
def cam_box(cam, ci):
    # same algorithm as the app: min-max normalised class map, hot cells (>=.6) connected to the peak. cam: (gh, gw, classes) raw per-cell scores
    gh, gw, c = cam.shape
    P = cam[..., ci]; P = (P - P.min()) / (P.max() - P.min() + 1e-9)
    by, bx = np.unravel_index(P.argmax(), P.shape)
    th = 0.6; seen = np.zeros_like(P, bool); st = [(by, bx)]; seen[by, bx] = True
    x0 = x1 = bx; y0 = y1 = by
    while st:
        yy, xx = st.pop(); x0, x1, y0, y1 = min(x0, xx), max(x1, xx), min(y0, yy), max(y1, yy)
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                ny, nx = yy + dy, xx + dx
                if 0 <= ny < gh and 0 <= nx < gw and not seen[ny, nx] and P[ny, nx] >= th: seen[ny, nx] = True; st.append((ny, nx))
    w, h = min(1, (x1 + 1 - x0) / gw * 1.2), min(1, (y1 + 1 - y0) / gh * 1.2)
    cx, cy = (x0 + x1 + 1) / 2 / gw, (y0 + y1 + 1) / 2 / gh
    l, t = max(0, cx - w / 2), max(0, cy - h / 2)
    return l, t, min(w, 1 - l), min(h, 1 - t)

def draw(im, probs, cam, th):
    im = im.copy(); d = ImageDraw.Draw(im); W, Hh = im.size; found = []
    for c in DEFECTS:
        if probs[cid[c]] >= th[c]:
            l, t, w, h = cam_box(cam, cid[c]); found.append(f"{c} {probs[cid[c]]:.0%}")
            d.ellipse([l * W, t * Hh, (l + w) * W, (t + h) * Hh], outline=(255, 90, 31), width=max(3, W // 90))
            d.text((l * W + 4, t * Hh + 4), c, fill=(255, 255, 255), font=font(max(12, W // 22)), stroke_width=2, stroke_fill=(255, 90, 31))
    return im, found

rng = np.random.default_rng(3)
pick = []
for c in DEFECTS:
    ix = np.where(y == cid[c])[0]; pick += list(rng.choice(ix, size=min(4, len(ix)), replace=False))
cols = 4; rws = (len(pick) + cols - 1) // cols
fig, axs = plt.subplots(rws, cols, figsize=(3.2 * cols, 3.3 * rws))
for a in np.ravel(axs): a.axis("off")
for a, i in zip(np.ravel(axs), pick):
    im2, found = draw(items[i][0].resize((384, 384)), oof[i], oof_cam[i], THR)
    a.imshow(im2); a.set_title(f"true: {classes[y[i]]}\nmodel: {', '.join(found) or 'no flag'}", fontsize=8)
plt.tight_layout(); plt.show()

## 7. Train the final model on all photos

In [ ]:
Xall, Yall, _ = make_set(np.arange(len(items)), N_AUG_TRAIN, True, 999)
final = fit(Xall, Yall, verbose=1)
final.export("saved_model")
del Xall

## 8. Export to LiteRT (.tflite) and check it matches

In [ ]:
conv = tf.lite.TFLiteConverter.from_saved_model("saved_model")
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]        # smaller file, float32 inputs/outputs
tfl = conv.convert()
open("defects.tflite", "wb").write(tfl)
print("defects.tflite: %.1f MB" % (len(tfl) / 1e6))

interp = tf.lite.Interpreter(model_content=tfl); interp.allocate_tensors()
inp = interp.get_input_details()[0]; outs = interp.get_output_details()
print("input", inp["shape"], inp["dtype"])
for o in outs: print("output", o["name"], o["shape"])
ops = sorted({d["op_name"] for d in interp._get_ops_details()})
print("ops used:", ops)
SUPPORTED = {"CONV_2D", "DEPTHWISE_CONV_2D", "ADD", "MUL", "HARD_SWISH", "MEAN", "FULLY_CONNECTED", "SOFTMAX", "DEQUANTIZE"}
print("OK: every op is supported by the CivilVerify app runner" if set(ops) <= SUPPORTED else "NOTE: ops the app runner may not support: %s . Send this output to Claude." % sorted(set(ops) - SUPPORTED))

def run_tfl(arr):
    interp.set_tensor(inp["index"], arr[None].astype(np.float32)); interp.invoke()
    p = cam = None
    for o in outs:
        v = interp.get_tensor(o["index"])
        if v.ndim == 4: cam = v[0]
        else: p = v[0]
    return p, cam

diff = 0.0
for a in Xt if False else [full_view(items[i][0]) for i in range(0, len(items), max(1, len(items) // 10))]:
    pk, ck = final.predict(a[None].astype(np.float32), verbose=0)
    pt, ct = run_tfl(a)
    diff = max(diff, float(np.abs(pk[0] - pt).max()), float(np.abs(ck[0] - ct).max()) / 10)
print("max difference Keras vs tflite: %.4f (should be below about 0.02)" % diff)

In [ ]:
rows, good_flagged = counts(THR)
card = {
  "model": "MobileNetV3Small transfer learning, fully convolutional head", "created": datetime.date.today().isoformat(),
  "classes": classes, "thresholds": THR,
  "input": {"shape": [1, IMG, IMG, 3], "dtype": "float32", "range": "0-255 raw pixels, RGB, NHWC"},
  "outputs": "class probabilities [1,classes] and per-cell class scores [1,%d,%d,classes] (raw logits)" % (IMG // 32, IMG // 32),
  "dataset": {c: int((y == cid[c]).sum()) for c in classes}, "independent_groups": int(len(set(group))),
  "cross_validation": {
    "folds": FOLDS, "top_class_correct_percent": round(100 * float((pred == y).mean()), 1),
    "auc": {c: round(a, 3) for c, a in aucs.items()},
    "per_defect": {c: f"caught {r['caught']}/{r['total']}; good photos flagged {r['on_good']}; other-defect photos also flagged {r['on_other_defects']}" for c, r in rows.items()},
    "good_flagged_with_any_defect": f"{good_flagged}/{ngood}",
    "confusion_matrix": cm.values.tolist()},
  "localisation": "Circles come from a class activation map learned from whole-photo labels only. They have not been measured against hand-drawn boxes.",
  "notes": "Small, partly stock-photo dataset. Not validated on photos from a real site. Screening aid only; an engineer decides."
}
json.dump(card, open("defects_card.json", "w"), indent=2); print(json.dumps(card, indent=2))

## 9. Try new photos (optional): probabilities and circles exactly as the app will show them

In [ ]:
up2 = files.upload()
for name, data in up2.items():
    im = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert("RGB")
    p, cam = run_tfl(full_view(im))
    shown, found = draw(im.resize((512, int(512 * im.height / im.width))), p, cam, THR)
    print(name, {c: round(float(p[cid[c]]), 2) for c in classes}, "->", ", ".join(found) or "no flag")
    plt.figure(figsize=(5, 5)); plt.imshow(shown); plt.axis("off"); plt.show()

In [ ]:
files.download("defects.tflite"); files.download("defects_card.json")

**Next:** send `defects.tflite` and `defects_card.json` back to Claude to be wired in, or place both in the `model/` folder of your GitHub repo (the app already looks for them).
Quote the `cross_validation` section of `defects_card.json` in your report, together with the dataset size and the limitation notes.